# Module 5: Gaussian Process Fundamentals

## Learning Objectives
By completing this notebook, you will:
1. Understand Gaussian Processes as distributions over functions
2. Implement GP regression from scratch using numpy
3. Build GP models in PyMC for commodity price forecasting
4. Explore how kernel parameters control function properties
5. Apply GPs to real commodity price data

## Prerequisites
- Multivariate normal distribution
- Bayesian inference fundamentals
- Linear algebra (matrix operations)

## Estimated Time: 75 minutes

---

In [ ]:
learning_objectives(["Understand Gaussian Processes as distributions over functions", "Implement GP regression from scratch using numpy", "Build GP models in PyMC for commodity price forecasting", "Explore how kernel parameters control function properties", "Apply GPs to real commodity price data", "Multivariate normal distribution"])

In [ ]:
import sys; sys.path.insert(0, '../../../../..')
from resources.notebook_style import apply_course_theme, learning_objectives, section_divider, callout, key_takeaways
from resources.graphics.plot_theme import apply_plot_theme

apply_course_theme()
apply_plot_theme()

## Setup & Imports

In [ ]:
# Purpose: Import required libraries for GP modeling
# Key Concept: GPs require numerical linear algebra and Bayesian tools

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pymc as pm
import arviz as az
from scipy import stats
from scipy.spatial.distance import cdist

# Set random seeds
np.random.seed(42)
az.style.use('arviz-darkgrid')

# Plotting configuration
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

print(f"PyMC version: {pm.__version__}")
print(f"NumPy version: {np.__version__}")

## Conceptual Introduction

### What is a Gaussian Process?

**Key Insight:** Instead of choosing a specific functional form (linear, polynomial, exponential), a GP places a **prior directly on the space of functions**.

**Formal Definition:**
A Gaussian Process is a collection of random variables, any finite number of which have a joint Gaussian distribution.

$$f(\mathbf{x}) \sim \mathcal{GP}(m(\mathbf{x}), k(\mathbf{x}, \mathbf{x}'))$$

Where:
- $m(\mathbf{x})$ is the **mean function** (expected value of $f$ at $\mathbf{x}$)
- $k(\mathbf{x}, \mathbf{x}')$ is the **kernel** (covariance between $f(\mathbf{x})$ and $f(\mathbf{x}')$)

### Visual Intuition

```
Before seeing data (GP Prior):
      f(x)
       │     ╱╲     ╱╲
       │    ╱  ╲   ╱  ╲         Sample 1
       │   ╱    ╲ ╱    ╲
       │  ╱      ╳      ╲
       │ ╱      ╱ ╲      ╲
       │╱      ╱   ╲      ╲     Sample 2
       └──────────────────────▶ x

After seeing data (GP Posterior):
      f(x)
       │         ●              (Observed points)
       │     ╱╲  │  ╱╲
       │    ╱  ╲ │ ╱  ╲         Posterior mean
       │   ╱    ╲│╱    ╲
       │  ╱      ●      ╲       (Function "pinned" at data)
       │ ╱       │       ╲
       └─────────●──────────▶ x
```

### Why GPs for Commodity Forecasting?

1. **Nonparametric flexibility** - No need to specify functional form
2. **Uncertainty quantification** - Automatic prediction intervals
3. **Kernel design** - Encode domain knowledge (seasonality, smoothness)
4. **Data efficiency** - Work well with limited data

## Part 1: GP from Scratch

### Squared Exponential (RBF) Kernel

The most common kernel:

$$k(x, x') = \sigma^2 \exp\left(-\frac{(x - x')^2}{2\ell^2}\right)$$

**Parameters:**
- $\sigma^2$: Signal variance (amplitude of function)
- $\ell$: Length scale (how quickly correlation decays with distance)

In [ ]:
# Purpose: Implement RBF kernel from scratch
# Key Concept: Kernel encodes similarity between points

def rbf_kernel(X1, X2, sigma=1.0, length_scale=1.0):
    """
    Squared Exponential (RBF) kernel.
    
    Parameters
    ----------
    X1 : array-like, shape (n1, d)
        First set of points
    X2 : array-like, shape (n2, d)
        Second set of points
    sigma : float
        Signal standard deviation
    length_scale : float
        Length scale parameter
    
    Returns
    -------
    K : array, shape (n1, n2)
        Kernel matrix
    """
    # Compute pairwise squared distances
    sq_dists = cdist(X1, X2, 'sqeuclidean')
    
    # Apply RBF formula
    K = sigma**2 * np.exp(-sq_dists / (2 * length_scale**2))
    
    return K

# Test the kernel
X_test = np.array([[0], [1], [2], [3]])
K_test = rbf_kernel(X_test, X_test, sigma=1.0, length_scale=1.0)

print("RBF Kernel matrix:")
print(K_test)
print(f"\nNote: Diagonal elements = {K_test[0, 0]:.3f} (variance at each point)")
print(f"Nearby points (distance=1): {K_test[0, 1]:.3f}")
print(f"Far points (distance=3): {K_test[0, 3]:.3f}")
print("\nCorrelation decreases exponentially with distance.")

In [ ]:
# Purpose: Visualize how kernel parameters affect correlation structure
# Key Concept: Length scale controls smoothness

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Reference point
x_ref = np.array([[0.0]])
x_range = np.linspace(-5, 5, 100)[:, None]

# Plot 1: Effect of length scale
ax = axes[0]
for ell in [0.5, 1.0, 2.0]:
    K = rbf_kernel(x_ref, x_range, sigma=1.0, length_scale=ell)
    ax.plot(x_range, K.flatten(), label=f'ℓ = {ell}', linewidth=2)
ax.set_xlabel('Distance from x=0')
ax.set_ylabel('Correlation k(0, x)')
ax.set_title('Effect of Length Scale', fontsize=12, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
ax.axhline(0, color='black', linewidth=0.8)

# Plot 2: Effect of signal variance
ax = axes[1]
for sigma in [0.5, 1.0, 2.0]:
    K = rbf_kernel(x_ref, x_range, sigma=sigma, length_scale=1.0)
    ax.plot(x_range, K.flatten(), label=f'σ = {sigma}', linewidth=2)
ax.set_xlabel('Distance from x=0')
ax.set_ylabel('Covariance k(0, x)')
ax.set_title('Effect of Signal Variance', fontsize=12, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
ax.axhline(0, color='black', linewidth=0.8)

# Plot 3: Full covariance matrix visualization
ax = axes[2]
X_grid = np.linspace(0, 5, 20)[:, None]
K_full = rbf_kernel(X_grid, X_grid, sigma=1.0, length_scale=1.0)
im = ax.imshow(K_full, cmap='RdBu_r', origin='lower', aspect='auto')
ax.set_xlabel('Point index')
ax.set_ylabel('Point index')
ax.set_title('Full Covariance Matrix', fontsize=12, fontweight='bold')
plt.colorbar(im, ax=ax, label='Covariance')

plt.tight_layout()
plt.show()

print("\nInterpretation:")
print("- Larger ℓ → wider correlation, smoother functions")
print("- Larger σ → higher amplitude, more variation")
print("- Covariance matrix is symmetric and positive definite")

In [ ]:
# Purpose: Sample functions from GP prior
# Key Concept: Different hyperparameters yield different function types

def sample_gp_prior(X, kernel_fn, n_samples=3, **kernel_params):
    """
    Draw function samples from GP prior.
    
    Parameters
    ----------
    X : array, shape (n, 1)
        Input points
    kernel_fn : callable
        Kernel function
    n_samples : int
        Number of function samples
    **kernel_params : dict
        Kernel parameters
    
    Returns
    -------
    samples : array, shape (n_samples, n)
        Function samples
    """
    # Compute covariance matrix
    K = kernel_fn(X, X, **kernel_params)
    
    # Add small jitter for numerical stability
    K += 1e-8 * np.eye(len(X))
    
    # Sample from multivariate normal
    mean = np.zeros(len(X))
    samples = np.random.multivariate_normal(mean, K, size=n_samples)
    
    return samples

# Generate samples with different parameters
X_plot = np.linspace(0, 10, 100)[:, None]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

configs = [
    {'sigma': 1.0, 'length_scale': 0.5, 'title': 'Short Length Scale (ℓ=0.5)\nRough, Wiggly'},
    {'sigma': 1.0, 'length_scale': 2.0, 'title': 'Long Length Scale (ℓ=2.0)\nSmooth'},
    {'sigma': 0.5, 'length_scale': 1.0, 'title': 'Low Amplitude (σ=0.5)\nSmall Variations'},
    {'sigma': 2.0, 'length_scale': 1.0, 'title': 'High Amplitude (σ=2.0)\nLarge Variations'}
]

for ax, config in zip(axes.flatten(), configs):
    title = config.pop('title')
    samples = sample_gp_prior(X_plot, rbf_kernel, n_samples=5, **config)
    
    for i, sample in enumerate(samples):
        ax.plot(X_plot, sample, alpha=0.7, linewidth=2, label=f'Sample {i+1}')
    
    ax.set_xlabel('x')
    ax.set_ylabel('f(x)')
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.grid(True, alpha=0.3)
    ax.axhline(0, color='black', linewidth=0.8)
    if ax == axes[0, 0]:
        ax.legend(loc='upper right')

plt.tight_layout()
plt.show()

print("\nObservations:")
print("- Length scale controls smoothness (larger ℓ → smoother)")
print("- Signal variance controls amplitude (larger σ → bigger swings)")
print("- All samples are continuous (RBF is infinitely differentiable)")

## Part 2: GP Regression from Scratch

### Posterior Distribution

Given observations $(\mathbf{X}, \mathbf{y})$ and test points $\mathbf{X}_*$:

**Posterior Mean:**
$$\bar{f}_* = K_*^T (K + \sigma_n^2 I)^{-1} \mathbf{y}$$

**Posterior Covariance:**
$$\text{Cov}(f_*) = K_{**} - K_*^T (K + \sigma_n^2 I)^{-1} K_*$$

Where:
- $K = k(\mathbf{X}, \mathbf{X})$ (training covariance)
- $K_* = k(\mathbf{X}, \mathbf{X}_*)$ (train-test covariance)
- $K_{**} = k(\mathbf{X}_*, \mathbf{X}_*)$ (test covariance)
- $\sigma_n^2$ (observation noise)

In [ ]:
# Purpose: Implement GP posterior prediction
# Key Concept: Conditioning the prior on observed data

def gp_posterior(X_train, y_train, X_test, kernel_fn, sigma_n=0.1, **kernel_params):
    """
    Compute GP posterior mean and covariance.
    
    Parameters
    ----------
    X_train : array, shape (n, d)
        Training inputs
    y_train : array, shape (n,)
        Training outputs
    X_test : array, shape (m, d)
        Test inputs
    kernel_fn : callable
        Kernel function
    sigma_n : float
        Observation noise std
    **kernel_params : dict
        Kernel parameters
    
    Returns
    -------
    mu_post : array, shape (m,)
        Posterior mean
    cov_post : array, shape (m, m)
        Posterior covariance
    """
    # Compute kernel matrices
    K = kernel_fn(X_train, X_train, **kernel_params)  # n × n
    K_s = kernel_fn(X_train, X_test, **kernel_params)  # n × m
    K_ss = kernel_fn(X_test, X_test, **kernel_params)  # m × m
    
    # Add observation noise to training covariance
    K_y = K + sigma_n**2 * np.eye(len(X_train))
    
    # Solve linear system (more stable than explicit inverse)
    # K_y @ alpha = y_train  =>  alpha = K_y^{-1} @ y_train
    alpha = np.linalg.solve(K_y, y_train)
    
    # Posterior mean: K_s^T @ alpha
    mu_post = K_s.T @ alpha
    
    # Posterior covariance: K_ss - K_s^T @ K_y^{-1} @ K_s
    v = np.linalg.solve(K_y, K_s)
    cov_post = K_ss - K_s.T @ v
    
    return mu_post, cov_post

# Test with synthetic data
np.random.seed(42)
n_train = 20
X_train = np.sort(np.random.uniform(0, 10, n_train))[:, None]
y_true = np.sin(X_train[:, 0])
y_train = y_true + np.random.normal(0, 0.2, n_train)

# Predict
X_test = np.linspace(-1, 11, 200)[:, None]
mu_post, cov_post = gp_posterior(
    X_train, y_train, X_test,
    kernel_fn=rbf_kernel,
    sigma_n=0.2,
    sigma=1.0,
    length_scale=1.0
)

# Extract std from diagonal of covariance
std_post = np.sqrt(np.diag(cov_post))

print("GP posterior computed successfully.")
print(f"Posterior mean shape: {mu_post.shape}")
print(f"Posterior covariance shape: {cov_post.shape}")

In [ ]:
# Purpose: Visualize GP posterior
# Key Concept: Uncertainty quantification via credible intervals

fig, ax = plt.subplots(figsize=(12, 6))

# Plot training data
ax.scatter(X_train, y_train, c='red', s=80, zorder=10, 
          edgecolor='black', linewidth=1.5, label='Training Data')

# Plot true function
ax.plot(X_test, np.sin(X_test), 'g--', linewidth=2, 
       label='True Function: sin(x)', alpha=0.7)

# Plot posterior mean
ax.plot(X_test, mu_post, 'b-', linewidth=3, label='GP Posterior Mean')

# Plot credible intervals
ax.fill_between(
    X_test.flatten(),
    mu_post - 2 * std_post,
    mu_post + 2 * std_post,
    alpha=0.3,
    color='blue',
    label='95% Credible Interval'
)

# Styling
ax.set_xlabel('x', fontsize=12)
ax.set_ylabel('f(x)', fontsize=12)
ax.set_title('Gaussian Process Regression (From Scratch)', fontsize=14, fontweight='bold')
ax.legend(loc='upper right', fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_xlim(-1, 11)

plt.tight_layout()
plt.show()

print("\nKey observations:")
print("1. Posterior mean passes through (or near) observed data")
print("2. Uncertainty is LOW near training data")
print("3. Uncertainty INCREASES away from training data (especially in extrapolation)")
print("4. GP smoothly interpolates between points")

## Exercise 5.1: Explore Length Scale Impact

**Task:** Fit GP models with three different length scales (ℓ = 0.5, 1.0, 3.0) to the training data. Create a comparison plot showing:
- Training data points
- Posterior mean for each length scale
- 95% credible intervals

**Expected Output:** 
- Small ℓ: Overfitting (wiggly, narrow CI near data)
- Medium ℓ: Good fit
- Large ℓ: Undersmoothing (too smooth, wide CI)

**Hints:**
<details>
<summary>Hint 1</summary>
Use the gp_posterior function with different length_scale values
</details>

<details>
<summary>Hint 2</summary>
Create subplots or overlapping plots with different colors for each ℓ
</details>

In [ ]:
# YOUR CODE HERE
# ---------------
# 1. Define three length scales
# 2. For each, compute GP posterior
# 3. Create comparison visualization

length_scales = [0.5, 1.0, 3.0]

# YOUR CODE: Implement comparison

your_answer = None  # Replace with your plot or analysis

In [ ]:
# AUTO-GRADED TESTS - Do not modify
# ----------------------------------
def test_exercise_5_1():
    assert your_answer is not None, "Don't forget to implement your solution!"
    
    # Verify that three different posteriors were computed
    # (Students should store results in a data structure)
    print("Exercise 5.1 passed!")
    print("Expected observation: Smaller ℓ → more wiggly fit, Larger ℓ → smoother fit")

test_exercise_5_1()

## Part 3: GP Regression in PyMC

Instead of fixing hyperparameters, we'll infer them from data using Bayesian inference.

In [ ]:
# Purpose: Build GP model in PyMC with learned hyperparameters
# Key Concept: Bayesian treatment of kernel parameters

with pm.Model() as gp_model:
    # Priors on kernel hyperparameters
    ell = pm.Gamma('ell', alpha=2, beta=1)  # Length scale
    sigma = pm.HalfNormal('sigma', sigma=2)  # Signal std
    sigma_n = pm.HalfNormal('sigma_n', sigma=0.5)  # Noise std
    
    # Define covariance function
    cov = sigma**2 * pm.gp.cov.ExpQuad(1, ls=ell)
    
    # GP prior on latent function
    gp = pm.gp.Marginal(cov_func=cov)
    
    # Likelihood (with observation noise)
    y_obs = gp.marginal_likelihood('y_obs', X=X_train, y=y_train, sigma=sigma_n)
    
    # Sample from posterior
    trace_gp = pm.sample(
        1000,
        tune=1000,
        target_accept=0.9,
        return_inferencedata=True,
        random_seed=42
    )

print("\nPosterior summary of hyperparameters:")
print(az.summary(trace_gp, var_names=['ell', 'sigma', 'sigma_n'], round_to=3))

In [ ]:
# Purpose: Generate posterior predictions
# Key Concept: Conditional GP given observed data and learned hyperparameters

with gp_model:
    # Conditional GP for test points
    f_pred = gp.conditional('f_pred', X_test)
    
    # Sample from posterior predictive
    ppc_gp = pm.sample_posterior_predictive(
        trace_gp,
        var_names=['f_pred'],
        random_seed=42
    )

# Extract predictions
f_mean_pymc = ppc_gp.posterior_predictive['f_pred'].mean(dim=['chain', 'draw']).values
f_std_pymc = ppc_gp.posterior_predictive['f_pred'].std(dim=['chain', 'draw']).values

print("Posterior predictive samples generated.")

In [ ]:
# Purpose: Compare PyMC GP to manual implementation
# Key Concept: Learned hyperparameters vs. fixed values

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Plot 1: PyMC GP (learned hyperparameters)
ax = axes[0]
ax.scatter(X_train, y_train, c='red', s=80, zorder=10, 
          edgecolor='black', linewidth=1.5, label='Training Data')
ax.plot(X_test, np.sin(X_test), 'g--', linewidth=2, 
       label='True Function', alpha=0.7)
ax.plot(X_test, f_mean_pymc, 'b-', linewidth=3, label='GP Mean (PyMC)')
ax.fill_between(
    X_test.flatten(),
    f_mean_pymc - 2 * f_std_pymc,
    f_mean_pymc + 2 * f_std_pymc,
    alpha=0.3,
    color='blue',
    label='95% CI'
)
ax.set_xlabel('x', fontsize=12)
ax.set_ylabel('f(x)', fontsize=12)
ax.set_title('PyMC GP (Learned Hyperparameters)', fontsize=12, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_xlim(-1, 11)

# Plot 2: Hyperparameter posteriors
ax = axes[1]
az.plot_posterior(
    trace_gp,
    var_names=['ell', 'sigma', 'sigma_n'],
    ax=ax
)
ax.set_title('Posterior Distributions of Hyperparameters', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

# Extract learned values
ell_mean = trace_gp.posterior['ell'].mean().item()
sigma_mean = trace_gp.posterior['sigma'].mean().item()
sigma_n_mean = trace_gp.posterior['sigma_n'].mean().item()

print(f"\nLearned hyperparameters:")
print(f"Length scale (ℓ): {ell_mean:.3f}")
print(f"Signal std (σ): {sigma_mean:.3f}")
print(f"Noise std (σ_n): {sigma_n_mean:.3f}")
print(f"\nCompare to true noise: 0.200")

## Part 4: Application to Commodity Prices

Let's apply GP regression to simulated crude oil prices with realistic features:
- Long-term trend
- Volatility clustering
- Missing data points

In [ ]:
# Purpose: Generate synthetic crude oil price data
# Key Concept: Realistic commodity price dynamics

np.random.seed(42)
n_days = 500
t = np.arange(n_days)

# Components
trend = 60 + 0.02 * t  # Gradual upward trend
seasonal = 5 * np.sin(2 * np.pi * t / 365)  # Annual seasonality
noise = np.random.normal(0, 2, n_days)  # Random shocks

# Combine
oil_prices = trend + seasonal + noise

# Simulate missing data (e.g., weekends, holidays)
missing_idx = np.random.choice(n_days, size=100, replace=False)
observed_idx = np.setdiff1d(t, missing_idx)

X_oil_train = observed_idx[:, None].astype(float)
y_oil_train = oil_prices[observed_idx]

# Visualization
plt.figure(figsize=(12, 6))
plt.plot(t, oil_prices, 'gray', alpha=0.3, label='Complete Data (Hidden)')
plt.scatter(observed_idx, y_oil_train, c='blue', s=10, alpha=0.6, label='Observed Prices')
plt.xlabel('Days', fontsize=12)
plt.ylabel('Price ($/barrel)', fontsize=12)
plt.title('Simulated Crude Oil Prices with Missing Data', fontsize=14, fontweight='bold')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print(f"Total days: {n_days}")
print(f"Observed: {len(observed_idx)} ({100*len(observed_idx)/n_days:.1f}%)")
print(f"Missing: {len(missing_idx)} ({100*len(missing_idx)/n_days:.1f}%)")

## Exercise 5.2: Forecast Oil Prices with GP

**Task:** Build a PyMC GP model to:
1. Impute missing values in the historical period
2. Forecast 30 days into the future
3. Quantify uncertainty in both tasks

**Expected Output:**
- GP should smooth through noise while capturing trend
- Uncertainty should increase in forecast period
- Model should recover true prices in missing data regions

**Hints:**
<details>
<summary>Hint 1</summary>
Use pm.gp.Marginal with ExpQuad kernel
</details>

<details>
<summary>Hint 2</summary>
For prediction, create X_test that includes both missing historical points and future dates
</details>

<details>
<summary>Hint 3</summary>
Use gp.conditional() to make predictions at unobserved locations
</details>

In [ ]:
# YOUR CODE HERE
# ---------------
# 1. Build PyMC GP model
# 2. Sample from posterior
# 3. Generate predictions for missing + future data
# 4. Visualize results

# Test points: historical missing + 30 day forecast
X_oil_test = np.concatenate([
    missing_idx[:, None],
    np.arange(n_days, n_days + 30)[:, None]
]).astype(float)

with pm.Model() as oil_gp_model:
    # YOUR CODE: Build GP model
    pass

# YOUR CODE: Sample and predict

your_answer = None  # Replace with your trace

In [ ]:
# AUTO-GRADED TESTS - Do not modify
# ----------------------------------
def test_exercise_5_2():
    assert your_answer is not None, "Don't forget to implement your solution!"
    assert isinstance(your_answer, az.InferenceData), "Should return InferenceData"
    
    # Check that hyperparameters were learned
    assert 'ell' in your_answer.posterior or 'ls' in your_answer.posterior, \
        "Model should have length scale parameter"
    
    print("Exercise 5.2 passed!")
    print("GP successfully applied to commodity price forecasting")

test_exercise_5_2()

## Summary

### Key Takeaways

1. **GPs are distributions over functions** - Defined by mean and kernel functions

2. **Kernel encodes assumptions** - Smoothness, periodicity, and correlation structure

3. **Bayesian hyperparameter learning** - PyMC infers optimal kernel parameters from data

4. **Automatic uncertainty quantification** - Credible intervals narrow near data, widen away

5. **Flexible for commodity forecasting** - Handles missing data, nonlinear trends, and forecasting

### What's Next

**Next Notebook: Kernel Design** - Combine multiple kernels to capture complex commodity price patterns (trend + seasonality + shocks)

**Module 6: Advanced Inference** - Efficient sampling for GP models with large datasets

### Additional Resources

1. **Rasmussen & Williams (2006)**. *Gaussian Processes for Machine Learning* - Free online: [http://gaussianprocess.org/gpml/](http://gaussianprocess.org/gpml/)
2. **PyMC GP Tutorial**: [https://www.pymc.io/projects/examples/en/latest/gaussian_processes/](https://www.pymc.io/projects/examples/en/latest/gaussian_processes/)
3. **Kernel Cookbook**: [https://www.cs.toronto.edu/~duvenaud/cookbook/](https://www.cs.toronto.edu/~duvenaud/cookbook/)

---

*"GPs answer: given smoothness assumptions and observed data, what functions are plausible?"*